<a href="https://colab.research.google.com/github/Ahmed-Rafi07/-MongoDB-connection-string-in-the-.env-file-/blob/main/MISSING_DATA_LAB_Ecommerce.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(101)

# Generate baseline data
n_rows = 6200
data = {
    'order_id': range(10001, 10001 + n_rows),
    'order_type': np.random.choice(['Standard', 'Express', 'Next-Day'], size=n_rows, p=[0.7, 0.2, 0.1]),
    'region': np.random.choice(['North', 'South', 'East', 'West'], size=n_rows),
    'order_status': np.random.choice(['delivered', 'returned', 'shipped'], size=n_rows, p=[0.75, 0.21, 0.04]),
    'delivery_time_days': np.random.normal(loc=4.2, scale=1.5, size=n_rows).round(1),
    'product_rating': np.random.choice([1, 2, 3, 4, 5], size=n_rows, p=[0.05, 0.1, 0.25, 0.4, 0.2]),
    'customer_age': np.random.normal(loc=34.5, scale=11.2, size=n_rows).round(0)
}

df = pd.DataFrame(data)

# Ensure delivery times are strictly positive
df['delivery_time_days'] = df['delivery_time_days'].clip(lower=1.0)
df['customer_age'] = df['customer_age'].clip(lower=18.0)

# 1. Introduce 5.0% MCAR gaps in delivery_time_days (310 values)
mcar_delivery = np.random.choice(df.index, size=310, replace=False)
df.loc[mcar_delivery, 'delivery_time_days'] = np.nan

# 2. Introduce ~1.4% MCAR gaps in customer_age (88 values)
mcar_age = np.random.choice(df.index, size=88, replace=False)
df.loc[mcar_age, 'customer_age'] = np.nan

# 3. Introduce MAR gaps in product_rating (1,488 missing values, clustered on returned orders)
returned_indices = df[df['order_status'] == 'returned'].index
# Select exactly 1488 indices from returned orders to be missing
if len(returned_indices) >= 1488:
    mar_rating = np.random.choice(returned_indices, size=1488, replace=False)
else:
    # Fallback to make sure we hit exactly 1,488
    extra_needed = 1488 - len(returned_indices)
    extra_indices = np.random.choice(df.index.difference(returned_indices), size=extra_needed, replace=False)
    mar_rating = np.concatenate([returned_indices, extra_indices])

df.loc[mar_rating, 'product_rating'] = np.nan

# Verify setup
print("Dataset successfully initialized. Gaps present:")
print(df.isnull().sum()[df.isnull().sum() > 0])


Dataset successfully initialized. Gaps present:
delivery_time_days     310
product_rating        1488
customer_age            88
dtype: int64


In [3]:
print("Missing Value:")
print(
    df.isnull().sum()[
        df.isnull().sum()>0
    ]
  )

Missing Value:
delivery_time_days     310
product_rating        1488
customer_age            88
dtype: int64


In [4]:
delivery_before = df['delivery_time_days'].copy()

delivery_mean = df['delivery_time_days'].mean()

df['delivery_time_days'] = (
    df['delivery_time_days'].fillna(delivery_mean)
)

print("Mean used: ",round(delivery_mean,2))
print("Missing Value remaining: ",
      df['delivery_time_days'].isnull().sum()
      )

Mean used:  4.18
Missing Value remaining:  0


In [5]:
age_before = df['customer_age'].copy()

age_median = df['customer_age'].median()

df['customer_age'] = (
    df['customer_age'].fillna(age_median)
)
print("Median used: ",age_median)
print("Missing Value remaining: ", df['customer_age'].isnull().sum())

Median used:  34.0
Missing Value remaining:  0


In [6]:
df['rating_missing'] = (
    df['product_rating'].isnull().astype(int)
)

print(
    df.groupby('order_status')['rating_missing'].mean().round(3)
)

order_status
delivered    0.041
returned     1.000
shipped      0.061
Name: rating_missing, dtype: float64


In [7]:
df['product_rating'].fillna(...)

,product_rating
0,2.0
1,Ellipsis
2,2.0
3,3.0
4,4.0
...,...
6195,5.0
6196,2.0
6197,3.0
6198,4.0


In [8]:
df['Product_rating_missing'] = (
    df['product_rating'].isnull().astype(int)
)
print(
    df['Product_rating_missing'].value_counts()
)
print(
    "\nOriginal missing rating:",
    df['product_rating'].isnull().sum()
)

Product_rating_missing
0    4712
1    1488
Name: count, dtype: int64

Original missing rating: 1488


In [10]:
delivery_after = df['delivery_time_days']

before_Desc = delivery_before.describe()
after_desc = delivery_after.describe()

comparison = pd.DataFrame({
    'Before': before_Desc,
    'After': after_desc
})

comparison['Change'] = (
    comparison['After'] -
    comparison['Before']
)

print(comparison.round(3))

         Before     After   Change
count  5890.000  6200.000  310.000
mean      4.178     4.178    0.000
std       1.483     1.446   -0.038
min       1.000     1.000    0.000
25%       3.200     3.200    0.000
50%       4.200     4.178   -0.022
75%       5.200     5.100   -0.100
max      11.200    11.200    0.000


In [11]:
print("Final missing-value counts: ")

final_missing = df.isnull().sum()

print(
    final_missing[final_missing>0]
)

Final missing-value counts: 
product_rating    1488
dtype: int64
